# 1. 数据加载

导入所有依赖库，读取Kaggle比赛数据集。

In [1]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

In [ ]:

TRAIN_PATH = Path(r"E:\MLW\demo\train.csv")
TEST_PATH  = Path(r"E:\MLW\demo\test.csv")

# 读取原始数据
train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
print(f"训练集: {train_df.shape}, 测试集: {test_df.shape}")

训练集: (8693, 14), 测试集: (4277, 13)


# 2. 数据预处理

统一预处理，处理缺失值，软规则补全CryoSleep。

In [3]:
# 五项消费列名（后续多处复用）
SPEND_COLS = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]


def _split_or_missing(series, sep, n_parts):
    """拆分结构型字符串列；原值缺失时用占位符填充。"""
    placeholder = sep.join(["Missing"] * n_parts)
    return series.fillna(placeholder).astype(str).str.split(sep, expand=True)


def add_structural_features(df):
    """
    从PassengerId、Cabin、Name中提取结构特征：
    PassengerId -> GroupID / GroupPos
    Cabin       -> Deck / CabinNum / Side
    Name        -> LastName
    """
    out = df.copy()
    pid_parts = _split_or_missing(out["PassengerId"], "_", 2)
    out["GroupID"]  = pid_parts[0].astype(str)
    out["GroupPos"] = pd.to_numeric(pid_parts[1], errors="coerce")

    cabin_parts = _split_or_missing(out["Cabin"], "/", 3)
    out["Deck"]     = cabin_parts[0].astype(str)
    out["CabinNum"] = pd.to_numeric(cabin_parts[1], errors="coerce")
    out["Side"]     = cabin_parts[2].astype(str)

    out["Name"]     = out["Name"].fillna("Missing Missing").astype(str)
    out["LastName"] = out["Name"].str.split().str[-1].astype(str)
    out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
    for col in SPEND_COLS:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    return out


def apply_soft_cryo_imputation(df):
    """
    软规则补全CryoSleep：
    CryoSleep缺失 且 消费总额>0 => 补False
    （有消费记录的人不可能处于冷冻睡眠状态）
    """
    out = df.copy()
    raw_spend_sum = out[SPEND_COLS].fillna(0).sum(axis=1)
    mask = out["CryoSleep"].isna() & (raw_spend_sum > 0)
    out.loc[mask, "CryoSleep"] = False
    # 消费列NaN统一补0（冷冻状态或未消费）
    for col in SPEND_COLS:
        out[col] = out[col].fillna(0)
    return out

# 3. 特征工程

构造聚合特征（TotalSpend、NoSpend、AgeBin），并对剩余缺失值做兜底填充。

In [ ]:
# 年龄分箱边界（与CatBoost/LightGBM主线保持一致，做受控对比）
AGE_BIN_EDGES  = [-1.0, 12.0, 18.0, 25.0, 40.0, 60.0, 200.0]
AGE_BIN_LABELS = ["child", "teen", "young", "adult", "mid", "senior"]


def add_aggregate_features(df):
    """
    TotalSpend：五项消费之和（反映消费能力和冷冻状态）
    NoSpend   ：是否零消费（与CryoSleep强相关的二值特征）
    AgeBin    ：年龄分箱（减少连续年龄的过拟合风险）
    """
    out = df.copy()
    out["TotalSpend"] = out[SPEND_COLS].sum(axis=1)
    out["NoSpend"]    = (out["TotalSpend"] == 0).astype(int)
    out["Age"]        = out["Age"].fillna(out["Age"].median())
    out["AgeBin"] = pd.cut(
        out["Age"], bins=AGE_BIN_EDGES, labels=AGE_BIN_LABELS, include_lowest=True
    ).astype("object")
    return out


def final_fill(df):
    """兜底填补：数值列用中位数，类别列用字符串Missing。"""
    out = df.copy()
    for col in ["Age", "GroupPos", "CabinNum", "TotalSpend", "NoSpend"]:
        out[col] = pd.to_numeric(out[col], errors="coerce").fillna(out[col].median())
    for col in ["HomePlanet", "CryoSleep", "Destination", "VIP",
                "GroupID", "Deck", "Side", "LastName", "AgeBin"]:
        out[col] = out[col].fillna("Missing").astype(str)
    return out


def prepare_train_test(train_df, test_df):
    """
    做特征工程：
    保证两边的类别值域、分箱边界、缺失填充口径完全一致。
    """
    train_tmp = train_df.copy(); train_tmp["_is_train"] = 1
    test_tmp  = test_df.copy();  test_tmp["_is_train"]  = 0
    test_tmp["Transported"] = float("nan")

    full = pd.concat([train_tmp, test_tmp], axis=0, ignore_index=True)
    full = add_structural_features(full)
    full = apply_soft_cryo_imputation(full)
    full = add_aggregate_features(full)
    full = final_fill(full)

    train_p = full[full["_is_train"] == 1].copy().drop(columns=["_is_train"])
    test_p  = full[full["_is_train"] == 0].copy().drop(columns=["_is_train"])

    # 19个核心特征（与CatBoost/LightGBM主线完全一致，实现受控对比）
    feature_cols = [
        "HomePlanet", "CryoSleep", "Destination", "Age", "VIP",
        "RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck",
        "GroupID", "GroupPos", "Deck", "CabinNum", "Side",
        "LastName", "TotalSpend", "NoSpend", "AgeBin",
    ]
    # XGBoost通过enable_categorical=True + pd.Categorical原生支持类别特征
    categorical_cols = [
        "HomePlanet", "CryoSleep", "Destination", "VIP",
        "GroupID", "Deck", "Side", "LastName", "AgeBin",
    ]
    return train_p, test_p, feature_cols, categorical_cols


# 执行特征工程
train_p, test_p, feature_cols, categorical_cols = prepare_train_test(train_df, test_df)
print(f"特征数: {len(feature_cols)}, 类别特征数: {len(categorical_cols)}")

特征数: 19, 类别特征数: 9


# 4. 模型配置

XGBoost超参数（受控对比基线：与CatBoost/LightGBM使用相同特征，**仅替换模型**）。

In [5]:
# XGBoost超参数（默认基线参数，未做Optuna调优）
XGB_PARAMS = dict(
    n_estimators      = 600,    # 最大树的数量
    learning_rate     = 0.05,   # 学习率（较小值+更多树=更稳定）
    max_depth         = 6,      # 树的最大深度
    min_child_weight  = 1,      # 叶节点最少样本权重（防止过拟合）
    subsample         = 0.9,    # 每棵树使用90%行（随机采样防过拟合）
    colsample_bytree  = 0.9,    # 每棵树使用90%特征
    reg_alpha         = 0.0,    # L1正则化系数
    reg_lambda        = 1.0,    # L2正则化系数
    eval_metric       = "logloss",   # early stopping 依赖此指标
    early_stopping_rounds = 100,      # 验证集100轮不提升则停止
    tree_method       = "hist",       # 必须为hist才能启用类别特征支持
    enable_categorical= True,          # 原生支持pd.Categorical列
    random_state      = 42,
    n_jobs            = -1,
)

# 交叉验证设置
N_SPLITS              = 5    # 5折交叉验证
RANDOM_STATE          = 42

print("模型: XGBoost (受控对比 CatBoost/LightGBM 主线，仅改模型其余不变)")
print(f"CV策略: {N_SPLITS}折 StratifiedKFold")

模型: XGBoost (受控对比 CatBoost/LightGBM 主线，仅改模型其余不变)
CV策略: 5折 StratifiedKFold


# 5. 训练

定义模型构建函数和类别列对齐工具函数。
XGBoost通过`enable_categorical=True` + `pd.Categorical`原生支持类别特征，
与LightGBM类似，需要对齐三份数据的类别空间。

In [6]:
def _align_categorical_levels(X_train, X_valid, X_test, categorical_cols):
    """
    对齐三份数据的类别列level集合。
    XGBoost enable_categorical模式要求类别列为pd.Categorical dtype，
    对齐类别空间确保三份数据的编码一致。
    """
    X_train, X_valid, X_test = X_train.copy(), X_valid.copy(), X_test.copy()
    for col in categorical_cols:
        all_vals   = pd.concat([X_train[col], X_valid[col], X_test[col]]).astype(str)
        categories = pd.Index(sorted(all_vals.unique().tolist()))
        X_train[col] = pd.Categorical(X_train[col].astype(str), categories=categories)
        X_valid[col] = pd.Categorical(X_valid[col].astype(str), categories=categories)
        X_test[col]  = pd.Categorical(X_test[col].astype(str),  categories=categories)
    return X_train, X_valid, X_test


def build_xgb_model():
    """构建XGBoost分类器实例（每折重新初始化）。"""
    return XGBClassifier(**XGB_PARAMS)

# 6. 交叉验证

5折StratifiedKFold：每折独立训练，用early stopping防过拟合，汇总OOF预测评估泛化能力。

In [7]:
def cross_validate_and_predict(train_p, test_p, feature_cols, categorical_cols):
    """
    5折分层交叉验证：
    - StratifiedKFold：保证每折正负比例与整体一致
    - early_stopping_rounds：验证集logloss 100轮不降则停止，避免过拟合
    - OOF(Out-of-Fold)：每条训练样本只在没见过它的折上预测，
      汇总后的OOF准确率是模型泛化能力的无偏估计
    - enable_categorical：XGBoost原生处理pd.Categorical类型列
    """
    X      = train_p[feature_cols].copy()
    y      = train_p["Transported"].astype(int).copy()
    X_test = test_p[feature_cols].copy()

    oof_proba        = np.zeros(len(train_p), dtype=float)
    test_proba_folds = []

    cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y), start=1):
        X_tr, y_tr = X.iloc[tr_idx].copy(), y.iloc[tr_idx].copy()
        X_va, y_va = X.iloc[va_idx].copy(), y.iloc[va_idx].copy()

        # 对齐类别列（转为pd.Categorical，统一类别空间）
        X_tr, X_va, X_te = _align_categorical_levels(
            X_tr, X_va, X_test, categorical_cols
        )

        model = build_xgb_model()
        model.fit(
            X_tr, y_tr,
            eval_set=[(X_va, y_va)],
            verbose=False,
        )

        best_iter = model.best_iteration
        va_proba  = model.predict_proba(X_va)[:, 1]
        te_proba  = model.predict_proba(X_te)[:, 1]

        fold_acc = accuracy_score(y_va, (va_proba >= 0.5).astype(int))
        oof_proba[va_idx] = va_proba
        test_proba_folds.append(te_proba)
        print(f"Fold {fold}: accuracy={fold_acc:.5f}, best_iter={best_iter}")

    # 测试集预测 = 5折概率的平均值
    test_proba = np.mean(np.vstack(test_proba_folds), axis=0)
    oof_pred   = (oof_proba >= 0.5).astype(int)
    oof_acc    = accuracy_score(y, oof_pred)

    print(f"\nCV 本地OOF accuracy: {oof_acc:.5f}")
    return y, oof_proba, oof_pred, test_proba


y_true, oof_proba, oof_pred, test_proba = cross_validate_and_predict(
    train_p, test_p, feature_cols, categorical_cols
)

Fold 1: accuracy=0.81254, best_iter=277
Fold 2: accuracy=0.80679, best_iter=327
Fold 3: accuracy=0.82116, best_iter=308
Fold 4: accuracy=0.82163, best_iter=348
Fold 5: accuracy=0.80035, best_iter=190

CV 本地OOF accuracy: 0.81249


# 7. 预测

将测试集概率用阈值0.5转为二分类标签，整理为提交格式。

In [8]:
# 概率>=0.5 => Transported=True，<0.5 => False
final_pred = (test_proba >= 0.5)

submission = pd.DataFrame({
    "PassengerId": test_p["PassengerId"].values,
    "Transported": final_pred,
})

print(f"预测Transported=True的比例: {final_pred.mean():.3f}")
submission.head()

预测Transported=True的比例: 0.507


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True


# 8. 提交

保存CSV文件，上传至Kaggle。

> XGBoost Baseline Kaggle LB = 0.80734
> （受控对比基线：19特征与CatBoost/LightGBM完全一致，仅替换模型）

In [9]:
# 保存提交文件
OUTPUT_PATH = "submission_xgboost_baseline.csv"
submission.to_csv(OUTPUT_PATH, index=False)
print(f"已保存: {OUTPUT_PATH}")
print(submission["Transported"].value_counts())

已保存: submission_xgboost_baseline.csv
Transported
True     2168
False    2109
Name: count, dtype: int64
